In [22]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import json
import numpy as np
import datetime
import matplotlib.pyplot as plt

THRESHOLD_TIMESTAMPS = 16

In [3]:
def extract_json(filename: str):
    with open(filename, "r") as f:
        for line in f:
            session = json.loads(line)
            yield session["session"], session["events"]
total_lenght = extract_json("train.jsonl")



In [ ]:
sessions = []

for i, (session_id, eventstotal) in enumerate(extract_json("train.jsonl")):
    aids, timestamps, events_type = [], [], []
    for event in eventstotal:
        aids.append(event["aid"])
        timestamps.append(event["ts"])
        events_type.append(event["type"])
        
    sessions.append({
            "session_id": i,
            "events": {
            "aid": aids,
            "timestamps": timestamps,
            "events_type": events_type    
            },
        })
    if i >= 100000:
        break

In [5]:
class OttoDataSetSession(Dataset):
    def __init__(self, session):
        self.session = session
        self.event_map = {"clicks":1, "carts": 2, "orders": 3}

    def __len__(self) -> int:
        return len(self.session)

    def __getitem__(self, index):
        session = self.session[index]
                 
        events = session["events"]
        
        aids = torch.tensor(events["aid"], dtype=torch.int64)
        
        timestamps = torch.tensor(events["timestamps"], dtype=torch.long)
        
        events_type = torch.tensor( [self.event_map[e] for e in events["events_type"]], dtype=torch.int64)
        return {
            "session_id": torch.tensor(session["session_id"], dtype=torch.int64),
            "aid": aids,
            "timestamps": timestamps,
            "type": events_type
        }
    

In [9]:
sessions_in_dataset = OttoDataSetSession(sessions)
print(f"Total len of the Sessions: {len(sessions_in_dataset)}")

session_sample_lenght = []
for i in range(len(sessions_in_dataset)):
    sample = sessions_in_dataset[i]["timestamps"]
    if len(sample) >= THRESHOLD_TIMESTAMPS:
        session_sample_lenght.append(sample)




Total len of the Sessions: 100001


In [65]:
avg_ofs_day_dataset = []
for i in range(len(session_sample_lenght)):
    ts = session_sample_lenght[i]
    min_ts = min(ts)
    max_ts = max(ts)
    avg = sum(ts) / len(ts)
    diff_seconds = float(max_ts - min_ts) / 1000
    
    avg_ofs_day_dataset.append(diff_seconds)
    diff = datetime.timedelta(seconds=diff_seconds)
    
    
    #print(f"This is the min TimeStamp of this Session {i} {(datetime.datetime.fromtimestamp(int(min_ts / 1000)).strftime('%d-%m-%Y %H:%M:%S'))}")
    #print(f"This is the max TimeStamp of this Session {i} {(datetime.datetime.fromtimestamp(int(max_ts) / 1000)).strftime('%d-%m-%Y %H:%M:%S')}")
    #print(f"This is the avg TimeStamp of this Session {i} {(datetime.datetime.fromtimestamp(int(avg) / 1000)).strftime('%d-%m-%Y %H:%M:%S')}")
    
    #print(f"Difference of days and everything {diff}")
    #print("=======================================================================")
    
    if i == 10000:
        break

In [67]:
#print(avg_ofs_day_dataset)
converted = [datetime.timedelta(seconds=x) for x in avg_ofs_day_dataset]
print(f"Total Avarage per 1000 of all the sessions {np.mean(converted)}")


Total Avarage per 1000 of all the sessions 21 days, 8:02:30.081120
